# Deterministic Execution: From Data to Code

This tutorial demonstrates the deterministic **Execution Engine** of `skforecast-ai` using a single-shot forecast.

While the LLM (Reasoning Engine) is great for exploring and diagnosing, the actual forecasting pipeline is 100% rule-based and transparent. We will profile a dataset, execute a forecast with probabilistic intervals, and then output the **exact, standalone Python script** that produced the results.

No black boxes. No hallucinated code.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
from skforecast_ai import ForecastingAssistant

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 4)

## 1. Load Real-World Data

We'll use the `h2o_exog` dataset, which models monthly corticosteroid sales alongside two exogenous variables.

In [ ]:
url = 'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/h2o_exog.csv'
data = pd.read_csv(url)

data['fecha'] = pd.to_datetime(data['fecha'])
data = data.set_index('fecha')
data = data.asfreq('MS')

ax = data['y'].plot(title='Monthly Sales (h2o_exog)')
ax.set_ylabel('Sales')
plt.show()

## 2. End-to-End Execution with Intervals

We initialize the assistant without an API key. In this default deterministic mode, the assistant runs entirely locally.

We will request an 80% prediction interval (`interval=[0.1, 0.9]`) to quantify our uncertainty over a 12-month horizon.

In [ ]:
assistant = ForecastingAssistant()  # Deterministic mode

# Profile, plan, generate code, and execute in one call
result = assistant.forecast(
             data        = data,
             target      = 'y',
             date_column = 'fecha',
             steps       = 12,
             interval    = [0.1, 0.9]
         )

print("Forecast Metrics (Training set estimation):")
display(result.metrics)

print("\nPredictions (First 5 steps):")
display(result.predictions.head())

Let's visualize the historical data along with our point predictions and uncertainty intervals.

In [ ]:
fig, ax = plt.subplots()

# Plot last 3 years of history
data['y'].iloc[-36:].plot(ax=ax, label='Historical Sales')

# Plot predictions
preds = result.predictions
preds['pred'].plot(ax=ax, label='Forecast', color='red')

# Fill intervals
ax.fill_between(
    preds.index,
    preds['lower_bound'],
    preds['upper_bound'],
    color='red',
    alpha=0.15,
    label='80% Interval'
)

ax.set_title('Sales Forecast with Prediction Intervals')
ax.legend()
plt.show()

## 3. The Fidelity Guarantee: `result.code`

The defining feature of the `skforecast-ai` Execution Engine is the **Fidelity Guarantee**.

The script stored in `result.code` is not an approximation. It is the literal Python string that was executed dynamically via `exec()` to produce the predictions and metrics you see above.

You can copy this code, save it to a `.py` file, and deploy it to production. It relies strictly on standard `skforecast` and `scikit-learn` libraries.

In [ ]:
print(result.code)

## 4. Supporting All Forecaster Types

While the default behavior uses `ForecasterRecursive`, `skforecast-ai` supports every major forecasting strategy in the `skforecast` ecosystem. You can easily switch between them using the `forecaster` argument.

Below, we demonstrate generating predictions for the same dataset using different underlying methodologies.

### ForecasterDirect
Trains a separate model for each step in the forecast horizon. Better for long horizons or when patterns change over time.

In [ ]:
result_direct = assistant.forecast(
                    data        = data,
                    target      = 'y',
                    date_column = 'fecha',
                    steps       = 12,
                    forecaster  = "ForecasterDirect",
                    estimator   = "Ridge"  # Recommended for direct strategy on small datasets
                )

display(result_direct.metrics)

### ForecasterStats (ARIMA)
Statistical models require no external estimator and provide native prediction intervals.

In [ ]:
result_stats = assistant.forecast(
                   data        = data,
                   target      = 'y',
                   date_column = 'fecha',
                   steps       = 12,
                   forecaster  = "ForecasterStats"
               )

display(result_stats.metrics)

### ForecasterFoundation (Zero-Shot)
Uses pre-trained foundation models (like Chronos-2). No training is performed; it uses the series context to forecast directly.

*(Note: Requires `pip install chronos-forecasting`)*

In [ ]:
# Uncomment to run if chronos-forecasting is installed
# result_foundation = assistant.forecast(
#     data        = data,
#     target      = 'y',
#     date_column = 'fecha',
#     steps       = 12,
#     forecaster  = "ForecasterFoundation",
# )
# display(result_foundation.metrics)